Sequential Chats and Customer Onboarding

In [1]:
!pip install autogen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 14.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [autogen]


In [3]:
import os
from getpass import getpass

In [4]:
api_key = getpass("Enter your OpenAI API key: ")

os.environ["OPENAI_API_KEY"] = api_key

Enter your OpenAI API key: ··········


In [5]:
from autogen import ConversableAgent
llm_config={"model": "gpt-4o"}

In [6]:
onboarding_personal_information_agent = ConversableAgent(
    name="Onboarding_Personal_Information_Agent",
    system_message='''You are a helpful customer onboarding agent,
    you are here to help new customers get started with our product.
    Your job is to gather customer's name and location.
    Do not ask for other information. Return 'TERMINATE'
    when you have gathered all the information.''',
    llm_config=llm_config,
    code_execution_config=False, #Prevent the agent from running any code, ensuring its outputs are always text-only.
    human_input_mode="NEVER", #The agent never requests human input and operates autonomously.
)

Suppose you are building a data analysis agent that needs to execute Python code to analyze a CSV file. In that situation, you may enable code execution with an appropriate executor configuration.

In [7]:
onboarding_topic_preference_agent = ConversableAgent(
    name="Onboarding_Topic_preference_Agent",
    system_message='''You are a helpful customer onboarding agent,
    you are here to help new customers get started with our product.
    Your job is to gather customer's preferences on news topics.
    Do not ask for other information.
    Return 'TERMINATE' when you have gathered all the information.''',
    llm_config=llm_config,
    code_execution_config=False, #Prevent the agent from running any code, ensuring its outputs are always text-only.
    human_input_mode="NEVER", #The agent never requests human input and operates autonomously.
)

code_execution_config=True, Useful for agent scenarios where mathematical calculation, data lookup/manipulation, or verifying intermediate reasoning steps is required (like a “Code Interpreter” or “Tool-Caller” agent).

In [8]:
customer_engagement_agent = ConversableAgent(
    name="Customer_Engagement_Agent",
    system_message='''You are a helpful customer service agent
    here to provide fun for the customer based on the user's
    personal information and topic preferences.
    This could include fun facts, jokes, or interesting stories.
    Make sure to make it engaging and fun!
    Return 'TERMINATE' when you are done.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
    is_termination_msg=lambda msg: "terminate" in msg.get("content").lower(), #If the agent's response contains the word TERMINATE, consider the conversation complete.
)

A Customer Proxy Agent, which acts as an interface between the real customer (human) and the other AI agents.

In [9]:
customer_proxy_agent = ConversableAgent(
    name="customer_proxy_agent",
    llm_config=False, #it is designed to obtain input directly from the human customer rather than generate AI responses.
    code_execution_config=False,
    human_input_mode="ALWAYS", #Human Input from Real customer
    is_termination_msg=lambda msg: "terminate" in msg.get("content").lower(),
)

Creating tasks

In [10]:
chats = [
    {
        "sender": onboarding_personal_information_agent,
        "recipient": customer_proxy_agent,
        "message":
            "Hello, I'm here to help you get started with our product."
            "Could you tell me your name and location?",
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt" : "Return the customer information "
                             "into as JSON object only: "
                             "{'name': '', 'location': ''}",
        },
        "max_turns": 2,
        "clear_history" : True
    },
    {
        "sender": onboarding_topic_preference_agent,
        "recipient": customer_proxy_agent,
        "message":
                "Great! Could you tell me what topics you are "
                "interested in reading about?",
        "summary_method": "reflection_with_llm",
        "max_turns": 1,
        "clear_history" : False
    },
    {
        "sender": customer_proxy_agent,
        "recipient": customer_engagement_agent,
        "message": "Let's find something fun to read.",
        "max_turns": 1,
        "summary_method": "reflection_with_llm",
    },
]

Start the onboarding process

In [11]:
from autogen import initiate_chats

chat_results = initiate_chats(chats)


********************************************************************************
Starting a new chat....

********************************************************************************
Onboarding_Personal_Information_Agent (to customer_proxy_agent):

Hello, I'm here to help you get started with our product.Could you tell me your name and location?

--------------------------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/autogen/agentchat/chat.py:69: UserWarning: Repetitive recipients detected: The chat history will be cleared by default if a recipient appears more than once. To retain the chat history, please set 'clear_history=False' in the configuration of the repeating agent.
  warnings.warn(


Replying as customer_proxy_agent. Provide feedback to Onboarding_Personal_Information_Agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation: Dr.Anand, Kottayam, Kerala
customer_proxy_agent (to Onboarding_Personal_Information_Agent):

Dr.Anand, Kottayam, Kerala

--------------------------------------------------------------------------------
Onboarding_Personal_Information_Agent (to customer_proxy_agent):

Thank you, Dr. Anand from Kottayam, Kerala. Your information has been recorded. If you need any further assistance, feel free to reach out. TERMINATE

--------------------------------------------------------------------------------
Replying as customer_proxy_agent. Provide feedback to Onboarding_Personal_Information_Agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation: Suggest good restaurants for seafoods in Kottayam
customer_proxy_agent (to Onboarding_Personal_Information_Agent):

Suggest good restaurants for seafoods 

In [12]:
for chat_result in chat_results:
    print(chat_result.summary)
    print("\n")

```json
{
  "name": "Dr. Anand",
  "location": "Kottayam, Kerala"
}
```


Dr. Anand, located in Kottayam, Kerala, is interested in reading about travel, nature, and trekking.


Dr. Anand, living in Kottayam, Kerala, is interested in reading about travel, nature, and trekking. Suggestions include exploring Chembra Peak in Kerala, understanding the origin of the word "trek," and learning about the twin phenomenon in Kodinhi. These topics are intended to inspire his interest in nature and exploration.




In [13]:
for chat_result in chat_results:
    print(chat_result.cost)
    print("\n")

{'usage_including_cached_inference': {'total_cost': 0.001245, 'gpt-4o-2024-08-06': {'cost': 0.001245, 'prompt_tokens': 258, 'completion_tokens': 60, 'total_tokens': 318}}, 'usage_excluding_cached_inference': {'total_cost': 0.001245, 'gpt-4o-2024-08-06': {'cost': 0.001245, 'prompt_tokens': 258, 'completion_tokens': 60, 'total_tokens': 318}}}


{'usage_including_cached_inference': {'total_cost': 0.00047250000000000005, 'gpt-4o-2024-08-06': {'cost': 0.00047250000000000005, 'prompt_tokens': 93, 'completion_tokens': 24, 'total_tokens': 117}}, 'usage_excluding_cached_inference': {'total_cost': 0.00047250000000000005, 'gpt-4o-2024-08-06': {'cost': 0.00047250000000000005, 'prompt_tokens': 93, 'completion_tokens': 24, 'total_tokens': 117}}}


{'usage_including_cached_inference': {'total_cost': 0.006347500000000001, 'gpt-4o-2024-08-06': {'cost': 0.006347500000000001, 'prompt_tokens': 647, 'completion_tokens': 473, 'total_tokens': 1120}}, 'usage_excluding_cached_inference': {'total_cost': 0.00634